### database connection and read data from parquet file

In [ ]:
import pandas as pd
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


df = pd.read_parquet("customerinfo_cluster.parquet")
df


### insert customers' `cluster`

In [ ]:

payload = df[["customer_id", "cluster"]].to_dict(orient="records")

with engine.begin() as conn:
    conn.execute(
        text("""
            UPDATE "CustomerInfo"
            SET customer_hierarchy = :cluster
            WHERE customer_id = :customer_id
        """),
        payload
    )

### insert `customer_type` by customers' `cluster`

In [ ]:
mapping = {
    0: "Promotional Sensitive Customers",
    1: "One-time Customers",
    2: "Lens Customers",
    3: "VIP Loyal Customers",
    4: "Regular Customers",
}

df = df[df["cluster"].notna()].copy()
df["customer_type"] = df["cluster"].map(mapping)

payload2 = df[["customer_id", "customer_type"]].to_dict(orient="records")
# payload2

In [ ]:
with engine.begin() as conn:
    conn.execute(
        text("""
            UPDATE "CustomerInfo"
            SET customer_type = :customer_type
            WHERE customer_id = :customer_id
        """),
        payload2
    )